In [0]:
#cleanup before execution

files = dbutils.fs.ls("/Volumes/pei/bronze/product/")
for f in files:
    dbutils.fs.rm(f.path, recurse=True)

files = dbutils.fs.ls("/Volumes/pei/bronze/customer/")
for f in files:
    dbutils.fs.rm(f.path, recurse=True)

files = dbutils.fs.ls("/Volumes/pei/bronze/orders/")
for f in files:
    dbutils.fs.rm(f.path, recurse=True)

In [0]:

dbutils.fs.rm("/Volumes/pei/bronze/product/_delta_log", recurse=True)

In [0]:
display(dbutils.fs.ls("/Volumes/pei/bronze/product/"))

In [0]:
display(dbutils.fs.ls("/Volumes/pei/bronze/orders/"))

In [0]:
display(dbutils.fs.ls("/Volumes/pei/bronze/customer/"))

In [0]:
# 1. Install dependency
%pip install pyyaml openpyxl


In [0]:
dbutils.library.restartPython()

In [0]:

# 2. Import libraries
import yaml
import pandas as pd

In [0]:
display(dbutils.fs.ls("/Volumes/PEI/raw/data/"))

In [0]:

# raw = (
#     spark.read.format("json")
#     .option("multiLine", "true")
#     .option("mode", "PERMISSIVE")
#     .load("dbfs:/Volumes/PEI/raw/data/Orders.json")
# )
# display(raw)
# # sink_path = "dbfs:/Volumes/pei/bronze/orders/"

# # print(f"Writing to: {sink_path}")

# # (
# #     raw.write
# #     .mode("overwrite")
# #     .format("parquet")
# #     .save(sink_path)
# # )

# # raw.show(2)

In [0]:


# # 3. Import your function (if modularized)
# # Option A: if using workspace files
# # from src.ingestion import read_data_sink

# # 4. Load config
# config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/bronze/orders_ingest.yaml"

# with open(config_path, "r") as f:
#     config = yaml.safe_load(f)

# # 5. Run pipeline
# for dataset in config["datasets"]:
#     read_data_sink(dataset)

In [0]:
# for dataset in config["datasets"]:
#     print(f"Processing: {dataset['name']}")
#     read_data_sink(dataset)

# CHANGES


In [0]:
def read_csv(spark, config):
    return (
        spark.read.format("csv")
        .options(**config.get("options", {}))
        .load(config["source_path"])
    )


def read_json(spark, config):
    return (
        spark.read.format("json")
        .options(**config.get("options", {}))
        .load(config["source_path"])
    )


def read_xlsx(spark, config):
    import pandas as pd

    df_pd = pd.read_excel(config["source_path"])

    # Fix problematic columns
    if "phone" in df_pd.columns:
        df_pd["phone"] = df_pd["phone"].astype(str)

    return spark.createDataFrame(df_pd)

In [0]:
READERS = {
    "csv": read_csv,
    "json": read_json,
    "parquet": lambda spark, config: spark.read.parquet(config["source_path"]),
    "xlsx": read_xlsx
}


def get_reader(format):
    if format not in READERS:
        raise ValueError(f"Unsupported format: {format}")
    return READERS[format]

In [0]:
config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/bronze/orders_ingest.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
def ingest_dataset(config):

    reader_fn = get_reader(config["format"])

    df = reader_fn(spark, config)

    df = standardize(df, config)

    write_to_bronze(df, config)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def standardize(df, config):
    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(config["source_path"]))
        .withColumn("dataset_name", lit(config["name"]))
    )

In [0]:
def write_to_bronze(df, config):
    target_path = config["target_path"]

    (
        df.write
        .mode(config.get("write_mode", "overwrite"))
        .format("parquet") 
        .save(target_path)
    )

In [0]:
# dbutils.fs.rm("dbfs:/Volumes/pei/bronze/product/", True)

In [0]:
for dataset in config["datasets"]:
    try:
        ingest_dataset(dataset)
    except Exception as e:
        print(f"Failed: {dataset['name']} → {str(e)}")

# SILVER

In [0]:
orders_df = spark.read.parquet("/Volumes/pei/bronze/orders/")
customers_df = spark.read.parquet("/Volumes/pei/bronze/customer/")
products_df = spark.read.parquet("/Volumes/pei/bronze/product/")

In [0]:
display(orders_df)

In [0]:
display(customers_df)

In [0]:
products_df.schema

In [0]:
display(products_df)

In [0]:
def process_table(config):
    df = read_data(config["source"])
    df = apply_schema(df, config["schema"])
    df = apply_transformations(df, config["transformations"])
    df = apply_dedup(df, config["deduplication"])
    df = select_columns(df, config["columns"])
    write_data(df, config["target"])

In [0]:
from pyspark.sql.functions import col

def apply_schema(df, schema_config):
    for field in schema_config:
        df = df.withColumn(
            field["name"],
            col(field["name"]).cast(field["type"])
        )
    return df

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def apply_dedup(df, dedup_config):
    if not dedup_config:
        return df

    keys = dedup_config["keys"]
    order_col = dedup_config["order_by"]

    window = Window.partitionBy(*keys).orderBy(col(order_col).desc())

    return (
        df.withColumn("rn", row_number().over(window))
          .filter("rn = 1")
          .drop("rn")
    )